In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first to confirm your environment is ready.
#
# This notebook uses only Python standard library modules (math, os, etc.)
# and will generate a file called water.log as part of the exercises.
# No additional package installations are required.

import sys, platform
print(f"Python {sys.version}")
print(f"Platform: {platform.system()}")
print("Environment ready — no additional packages required ✓")
print("Note: This notebook will write 'water.log' to your working directory.")


# Introduction to Python for Computational Chemistry
## Parsing Gaussian 16 Output Files

Welcome! This notebook is designed for undergraduate chemistry students who want to learn Python
in the context of computational chemistry. We will build up Python skills step-by-step and apply
each concept directly to reading and analyzing output from the **Gaussian 16** electronic-structure program.

### What you will learn
1. Core Python concepts: variables, strings, lists, loops, dictionaries, and file I/O
2. The structure of a Gaussian output file
3. How to extract **energetic** data: SCF energies, zero-point energies, enthalpies, and free energies
4. How to extract **geometric** data: Cartesian coordinates, bond lengths, and bond angles

> **Tip:** Run each code cell with **Shift + Enter**. Cells labeled *Practice* are for you to complete.


---
## Part 1: Python Fundamentals

Before we can parse a Gaussian output file we need a toolkit of Python concepts.
We will introduce each concept with a short explanation, show an example, and then give
you a practice exercise.


### 1.1 Variables and Data Types

Python stores information in **variables**. The four types you will use most in computational chemistry are:

| Type | Example | Typical use |
|------|---------|-------------|
| `int` | `6` | Atomic number, number of atoms |
| `float` | `-76.4187` | Energies in Hartree |
| `str` | `"oxygen"` | Element symbols, file names |
| `bool` | `True` | Flags (e.g., converged?) |

You do not need to declare a type — Python figures it out automatically.


In [ ]:
# ── Example 1.1 ─────────────────────────────────────────────────────
atomic_number   = 8          # int
scf_energy      = -76.4187   # float  (Hartree)
element_symbol  = "O"        # str
converged       = True       # bool

print(f"Element:        {element_symbol}")
print(f"Atomic number:  {atomic_number}")
print(f"SCF energy:     {scf_energy:.6f} Hartree")
print(f"Converged:      {converged}")
print()

# type() tells you the data type
print(f"type(scf_energy) → {type(scf_energy)}")


#### ✏️ Practice 1.1
Create variables to represent the following properties of a water molecule calculation, then print them
with descriptive labels:
- Number of atoms: 3
- Molecular formula as a string: `"H2O"`
- Final SCF energy (Hartree): -76.4187385461
- Calculation converged: True


In [ ]:
# ── Practice 1.1 ────────────────────────────────────────────────────
# Create your variables here

# n_atoms = ...
# formula = ...
# energy  = ...
# converged = ...

# Print each variable with a descriptive label


### 1.2 String Operations

Nearly everything in a text-based output file is read as a **string**.
The most important string operations for parsing are:

| Method | What it does |
|--------|-------------|
| `s.strip()` | Remove leading/trailing whitespace |
| `s.split()` | Split on whitespace → list of words |
| `s.split(sep)` | Split on a specific separator |
| `s.startswith(prefix)` | Returns `True` if string begins with prefix |
| `s.find(sub)` | Returns index of first match, or -1 |
| `sub in s` | Returns `True` if substring is present |

These will be our primary tools for hunting through Gaussian output.


In [ ]:
# ── Example 1.2 ─────────────────────────────────────────────────────
# A typical line from a Gaussian output file
line = " SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles"

print("Original line:")
print(repr(line))          # repr() shows hidden whitespace characters
print()

# strip() removes the leading space
clean = line.strip()
print("After strip():", repr(clean))
print()

# split() breaks the line on whitespace into a list of tokens
tokens = clean.split()
print("Tokens:", tokens)
print()

# We can check for a keyword to identify interesting lines
if "SCF Done:" in line:
    print("This is an SCF energy line!")
    # The energy is always the 5th token (index 4)
    energy = float(tokens[4])
    print(f"Energy = {energy:.10f} Hartree")


#### ✏️ Practice 1.2
The line below comes from the thermochemistry section of a Gaussian output file.
Extract the numerical value of the zero-point correction and convert it to kcal/mol.
*(Hint: 1 Hartree = 627.509 kcal/mol)*


In [ ]:
# ── Practice 1.2 ────────────────────────────────────────────────────
zpe_line = " Zero-point correction=                           0.020908 (Hartree/Particle)"
hartree_to_kcal = 627.509

# 1. Strip and split the line
# 2. Identify which token holds the number (print tokens to find out)
# 3. Convert to float and then to kcal/mol
# 4. Print the result



### 1.3 Lists and Loops

A **list** is an ordered, mutable collection. We will use lists to store things like:
- All lines of a file
- Coordinates of each atom
- A set of SCF energies across an optimization trajectory

A **`for` loop** iterates over every item in a list.


In [ ]:
# ── Example 1.3 ─────────────────────────────────────────────────────
# Atomic symbols and coordinates of water's standard orientation
symbols = ["O", "H", "H"]

# Each coordinate is itself a list [x, y, z]
coords = [
    [0.000000,  0.000000,  0.117176],
    [0.000000,  0.757306, -0.468706],
    [0.000000, -0.757306, -0.468706],
]

# Loop over atoms using zip to pair symbol + coordinate
print(f"{'Atom':<6} {'X':>10} {'Y':>10} {'Z':>10}")
print("-" * 40)
for symbol, xyz in zip(symbols, coords):
    print(f"{symbol:<6} {xyz[0]:>10.6f} {xyz[1]:>10.6f} {xyz[2]:>10.6f}")

print()
# len() gives the number of items
print(f"Number of atoms: {len(symbols)}")

# List indexing
print(f"Third atom: {symbols[2]}")       # index 2 = 3rd item (0-based)
print(f"Last atom:  {symbols[-1]}")      # -1 = last item


#### ✏️ Practice 1.3
Given the lists below, use a `for` loop to print the atomic number and mass of each element
in a nicely formatted table.


In [ ]:
# ── Practice 1.3 ────────────────────────────────────────────────────
elements       = ["H",  "C",   "N",   "O",   "F"]
atomic_numbers = [1,     6,     7,     8,     9]
atomic_masses  = [1.008, 12.011, 14.007, 15.999, 18.998]

# Print a table with columns: Symbol | Atomic Number | Atomic Mass
# Use a for loop (hint: zip can accept more than two lists)



### 1.4 Dictionaries

A **dictionary** (`dict`) maps *keys* to *values*. They are perfect for storing
structured data extracted from an output file, for example all thermochemical
quantities keyed by their name.

```python
d = {"key1": value1, "key2": value2}
d["key1"]          # access a value
d["key3"] = value3 # add a new key-value pair
```


In [ ]:
# ── Example 1.4 ─────────────────────────────────────────────────────
# Store thermochemistry results in a dictionary
thermo = {
    "scf_energy":    -76.4187385461,   # Hartree
    "zpe":             0.020908,        # Hartree
    "H_correction":    0.024686,
    "G_correction":    0.003610,
    "E_zpe":         -76.397830,
    "H_total":       -76.394052,
    "G_total":       -76.415128,
}

hartree_to_kcal = 627.509

print("Thermochemical summary for H₂O (B3LYP/6-31G*)")
print("=" * 50)
for label, value in thermo.items():
    print(f"  {label:<22}: {value:>16.6f} Hartree")

print()
print(f"Zero-point energy = {thermo['zpe'] * hartree_to_kcal:.3f} kcal/mol")


#### ✏️ Practice 1.4
Create a dictionary that maps each element symbol in water (`"O"`, `"H"`) to its
van der Waals radius in Å (`O = 1.52`, `H = 1.20`). Then write a loop that prints
each element and its radius.


In [ ]:
# ── Practice 1.4 ────────────────────────────────────────────────────
# Create the dictionary

# vdw_radii = { ... }

# Loop and print


### 1.5 Reading Files

All Gaussian output is stored in plain-text `.log` or `.out` files.
The standard Python pattern for reading a file is:

```python
with open("filename.log", "r") as fh:
    lines = fh.readlines()   # returns a list of strings, one per line
```

Using `with` ensures the file is closed automatically when you are done.

Each element of `lines` includes the newline character `\n` at the end —
`strip()` removes it.

First, let's write the sample Gaussian output to a file so we have something to work with.


In [ ]:
# ── Example 1.5a — write a sample output file ────────────────────────
gaussian_text = """
 Entering Gaussian System, Link 0=g16
 %chk=water.chk
 %mem=4GB
 %nprocshared=4
 ----------------------------------------------------------------------
 #p B3LYP/6-31G* Opt Freq

 Water molecule optimization and frequency calculation

 ----------------------------------------------------------------------
 Symbolic Z-matrix:
 Charge =  0 Multiplicity = 1
 O
 H  1  r1
 H  1  r1  2  a1

       Variables:
  r1                    0.96
  a1                  104.5


 GradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGrad

                          Input orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.000000
      2          1           0        0.000000    0.000000    0.960000
      3          1           0        0.927516    0.000000   -0.240000
 ---------------------------------------------------------------------

 SCF Done:  E(RB3LYP) =  -76.4083579211     A.U. after    9 cycles
            NFock= 9  Conv=0.47D-08     -V/T= 2.0089

 SCF Done:  E(RB3LYP) =  -76.4186251034     A.U. after   10 cycles
            NFock=10  Conv=0.38D-08     -V/T= 2.0091

 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
            NFock= 9  Conv=0.23D-08     -V/T= 2.0091

                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------

 Rotational constants (GHZ):         836.3077963         434.9876987         286.8513882

 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Energy=                    0.023742
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Energies=        -76.394996
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128

 E (Thermal)             CV                CP                S
 KCal/Mol        Cal/Mol-Kelvin    Cal/Mol-Kelvin    Cal/Mol-Kelvin
 14.896              6.002             8.314             44.278

 Zero-point vibrational energy     130943.2 (Joules/Mol)
                                    31.298 (Kcal/Mol)

 Frequencies --   1589.0765                3703.9127                3807.0960
 Red. masses --      1.0831                   1.0451                   1.0818
 Frc consts  --      1.6145                   8.4518                   9.2448
 IR Inten    --     67.1785                  10.4654                  49.3234

 Normal termination of Gaussian 16 at Mon Mar 14 12:34:56 2026.
"""

with open("water.log", "w") as fh:
    fh.write(gaussian_text)

print("File written: water.log")


In [ ]:
# ── Example 1.5b — read the file ─────────────────────────────────────
with open("water.log", "r") as fh:
    lines = fh.readlines()

print(f"Total lines read: {len(lines)}")
print()
print("First 5 lines:")
for i, line in enumerate(lines[:5]):
    print(f"  [{i}] {line}", end="")   # 'end=""' avoids double newline


#### ✏️ Practice 1.5
Read `water.log` and count:
1. The total number of lines in the file
2. The number of lines that contain the word `"SCF"`
3. The number of lines that start with `" Normal termination"`

Print all three counts.


In [ ]:
# ── Practice 1.5 ────────────────────────────────────────────────────
with open("water.log", "r") as fh:
    lines = fh.readlines()

# 1. Total lines (already done above — try it here for practice)
total = 0  # replace with correct expression

# 2. Lines containing "SCF"
scf_count = 0
# for line in lines:
#     if ...:
#         scf_count += 1

# 3. Lines starting with " Normal termination"
normal_term_count = 0

print(f"Total lines:              {total}")
print(f"Lines with 'SCF':         {scf_count}")
print(f"'Normal termination' lines: {normal_term_count}")


---
## Part 2: Anatomy of a Gaussian Output File

A Gaussian `.log` file is a sequential plain-text record of everything that happened during the calculation.
Key sections are summarised below.

```
Gaussian header (version, machine, job route)
Job title
Molecule specification (charge, multiplicity, Z-matrix or Cartesian input)
─────── Optimization loop (repeats until convergence) ────────────────────────
  Input orientation    ← Cartesian coordinates at this step
  SCF Done             ← Electronic energy at this geometry
  Converged? → if not, compute gradient and take a new step
──────────────────────────────────────────────────────────────────────────────
Standard orientation     ← Final optimized coordinates in principal-axis frame
Rotational constants
─────── Frequency / Thermochemistry section ──────────────────────────────────
  Zero-point correction
  Thermal corrections (Energy, Enthalpy, Gibbs)
  Sum of electronic + ZPE / H / G
  Harmonic vibrational frequencies
──────────────────────────────────────────────────────────────────────────────
Normal termination       ← ALWAYS verify this line is present!
```

> **Important:** If a Gaussian job crashes or runs out of time, `Normal termination`
> will be **absent**. Always check for it before using any results.

The atomic number look-up below will be useful later when we parse the coordinate block.


In [ ]:
# ── Atomic number → symbol look-up table ────────────────────────────
atomic_symbols = {
    1: "H",   2: "He",  3: "Li",  4: "Be",  5: "B",
    6: "C",   7: "N",   8: "O",   9: "F",  10: "Ne",
   11: "Na", 12: "Mg", 13: "Al", 14: "Si", 15: "P",
   16: "S",  17: "Cl", 18: "Ar", 19: "K",  20: "Ca",
}

# Test
print(atomic_symbols[8])    # oxygen
print(atomic_symbols[1])    # hydrogen


---
## Part 3: Extracting Energetic Information

### 3.1 SCF Energy

The SCF (Self-Consistent Field) energy is the electronic energy.
During a geometry optimisation Gaussian prints many `SCF Done` lines —
one per optimisation step. Usually you want the **last** one, which corresponds
to the converged geometry.

A typical `SCF Done` line looks like:

```
 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
```

The energy is **always** in position index 4 when the line is split on whitespace.


In [ ]:
# ── Example 3.1 — extract all SCF energies ──────────────────────────
with open("water.log", "r") as fh:
    lines = fh.readlines()

scf_energies = []    # will collect all values

for line in lines:
    if "SCF Done:" in line:
        tokens = line.split()
        energy = float(tokens[4])
        scf_energies.append(energy)

print("SCF energies found (Hartree):")
for i, e in enumerate(scf_energies, start=1):
    print(f"  Step {i}: {e:.10f}")

print()
print(f"Final (converged) SCF energy: {scf_energies[-1]:.10f} Hartree")


#### ✏️ Practice 3.1
Using the `scf_energies` list computed above:
1. Find and print the **lowest** SCF energy (the most stable point found).
2. Compute and print the **total energy lowering** from the first step to the last, in kcal/mol.
   *(1 Hartree = 627.509 kcal/mol)*


In [ ]:
# ── Practice 3.1 ────────────────────────────────────────────────────
hartree_to_kcal = 627.509

# scf_energies list is already available from the cell above

# 1. Lowest energy
# lowest = ...
# print(f"Lowest SCF energy: {lowest:.10f} Hartree")

# 2. Total energy lowering (first → last), in kcal/mol
# delta_E = ...
# print(f"Energy lowering:   {delta_E:.4f} kcal/mol")


### 3.2 Thermochemical Corrections

After a frequency calculation Gaussian prints a thermochemistry block:

```
 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Energy=                    0.023742
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Energies=        -76.394996
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128
```

We will use `startswith` and token indexing to extract each quantity.
Notice that the **sum quantities** are on lines starting with `" Sum of"` and
the numerical value is always the **last** token on the line.


In [ ]:
# ── Example 3.2 — extract thermochemistry ───────────────────────────
thermo_keys = {
    "Zero-point correction=":                          "zpe_correction",
    "Thermal correction to Energy=":                   "thermal_energy",
    "Thermal correction to Enthalpy=":                 "thermal_enthalpy",
    "Thermal correction to Gibbs Free Energy=":        "thermal_gibbs",
    "Sum of electronic and zero-point Energies=":      "E_zpe",
    "Sum of electronic and thermal Energies=":         "E_thermal",
    "Sum of electronic and thermal Enthalpies=":       "H_total",
    "Sum of electronic and thermal Free Energies=":    "G_total",
}

thermo = {}

with open("water.log", "r") as fh:
    for line in fh:
        stripped = line.strip()
        for search_str, key in thermo_keys.items():
            if stripped.startswith(search_str):
                value = float(stripped.split()[-1])   # last token
                thermo[key] = value

hartree_to_kcal = 627.509

print("Thermochemistry for H₂O  (B3LYP/6-31G*)")
print("=" * 55)
for key, val in thermo.items():
    print(f"  {key:<25}: {val:>16.6f} Hartree")

print()
print(f"ZPE            = {thermo['zpe_correction'] * hartree_to_kcal:.3f} kcal/mol")
print(f"Total enthalpy = {thermo['H_total']:.6f} Hartree")
print(f"Total Gibbs G  = {thermo['G_total']:.6f} Hartree")
print(f"−T·ΔS approx   = {(thermo['G_total'] - thermo['H_total']) * hartree_to_kcal:.3f} kcal/mol")


#### ✏️ Practice 3.2
Using the `thermo` dictionary:
1. Print the zero-point-corrected energy (`E_zpe`) and the Gibbs free energy (`G_total`) side by side in both Hartree and kcal/mol.
2. Compute the **thermal correction to the Gibbs free energy** relative to the zero-point-corrected energy (i.e., `G_total − E_zpe`) in kcal/mol.
   This quantity represents the finite-temperature entropic penalty.


In [ ]:
# ── Practice 3.2 ────────────────────────────────────────────────────
# thermo dict and hartree_to_kcal are available from above

# 1. Print E_zpe and G_total in both Hartree and kcal/mol
# ...

# 2. Thermal + entropic correction from ZPE to G (kcal/mol)
# delta = ...
# print(f"G_total - E_zpe = {delta:.3f} kcal/mol")


### 3.3 Checking for Normal Termination

This is a critical sanity check — **never use results from a job that did not terminate normally**.


In [ ]:
# ── Example 3.3 — verify job completed ──────────────────────────────
def check_normal_termination(filename):
    """Return True if 'Normal termination' appears in the file."""
    with open(filename, "r") as fh:
        content = fh.read()
    return "Normal termination" in content

if check_normal_termination("water.log"):
    print("✓ Job completed normally — results are trustworthy.")
else:
    print("✗ WARNING: Job did NOT terminate normally. Do not use these results!")


#### ✏️ Practice 3.3
Modify the function `check_normal_termination` so that it also returns the **timestamp** from
the Normal termination line (e.g., `"Mon Mar 14 12:34:56 2026"`). Print both the status and
the timestamp.

*Hint:* The line looks like:
```
 Normal termination of Gaussian 16 at Mon Mar 14 12:34:56 2026.
```
The date/time information starts at the word after `"at"`.


In [ ]:
# ── Practice 3.3 ────────────────────────────────────────────────────
def check_normal_termination_v2(filename):
    """
    Return (status, timestamp) where status is True/False and
    timestamp is the date/time string from the termination line (or None).
    """
    with open(filename, "r") as fh:
        lines = fh.readlines()

    for line in lines:
        if "Normal termination" in line:
            # Extract timestamp
            # tokens = line.split()
            # idx = tokens.index("at")
            # timestamp = " ".join(tokens[idx+1:]).rstrip(".")
            pass  # replace with your code

    return False, None  # replace with your return

status, timestamp = check_normal_termination_v2("water.log")
print(f"Normal termination: {status}")
print(f"Timestamp:          {timestamp}")


---
## Part 4: Extracting Geometric Information

### 4.1 Parsing the Standard Orientation Block

The **Standard orientation** block is printed after the final geometry is reached.
It contains the Cartesian coordinates of each atom in the principal-axis frame:

```
                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------
```

**Strategy:** scan line-by-line, set a flag when we see `"Standard orientation:"`,
skip the four header lines, then read coordinate lines until we hit the closing
dashes.


In [ ]:
# ── Example 4.1 — parse Standard orientation ────────────────────────
atomic_symbols = {
    1: "H",  6: "C",  7: "N",  8: "O",  9: "F",
   16: "S", 17: "Cl", 15: "P", 14: "Si",
}

def parse_standard_orientation(filename):
    """
    Parse the *last* Standard orientation block in a Gaussian log file.
    Returns a list of dicts: [{'symbol': 'O', 'x': ..., 'y': ..., 'z': ...}, ...]
    """
    atoms = []
    in_block = False
    header_lines_to_skip = 4   # two dashes + column header + dashes

    with open(filename, "r") as fh:
        lines = fh.readlines()

    for line in lines:
        # Start of a new Standard orientation block
        if "Standard orientation:" in line:
            in_block = True
            header_lines_to_skip = 4
            atoms = []   # reset so we keep only the LAST block
            continue

        if in_block:
            if header_lines_to_skip > 0:
                header_lines_to_skip -= 1
                continue
            # Closing dashes signal end of block
            if line.strip().startswith("-----"):
                in_block = False
                continue
            # Parse the coordinate line
            tokens = line.split()
            # tokens: [center#, atomic#, type, x, y, z]
            atomic_num = int(tokens[1])
            symbol = atomic_symbols.get(atomic_num, f"Z{atomic_num}")
            x, y, z = float(tokens[3]), float(tokens[4]), float(tokens[5])
            atoms.append({"symbol": symbol, "x": x, "y": y, "z": z})

    return atoms

# Run it
atoms = parse_standard_orientation("water.log")

print(f"{'#':<4} {'Symbol':<8} {'X (Å)':>10} {'Y (Å)':>10} {'Z (Å)':>10}")
print("-" * 46)
for i, atom in enumerate(atoms, start=1):
    print(f"{i:<4} {atom['symbol']:<8} {atom['x']:>10.6f} {atom['y']:>10.6f} {atom['z']:>10.6f}")


#### ✏️ Practice 4.1
Using `parse_standard_orientation`, write a function `count_by_element(atoms)` that
returns a dictionary of `{symbol: count}` for the atoms list. Apply it to the water
molecule and print the result.

Expected output:
```
{'O': 1, 'H': 2}
```


In [ ]:
# ── Practice 4.1 ────────────────────────────────────────────────────
def count_by_element(atoms):
    """Return a dict mapping element symbol to count."""
    counts = {}
    # for atom in atoms:
    #     symbol = atom["symbol"]
    #     ...
    return counts

# atoms is already defined from the example above
composition = count_by_element(atoms)
print("Molecular composition:", composition)


### 4.2 Computing Bond Lengths

Given two atoms with Cartesian coordinates $(x_1, y_1, z_1)$ and $(x_2, y_2, z_2)$,
the **Euclidean distance** is:

$$d = \sqrt{(x_2-x_1)^2 + (y_2-y_1)^2 + (z_2-z_1)^2}$$

This is the O–H bond length we will compute using the coordinates extracted above.


In [ ]:
# ── Example 4.2 — bond length ────────────────────────────────────────
import math

def bond_length(atom1, atom2):
    """
    Compute the distance (Å) between two atom dicts
    with keys 'x', 'y', 'z'.
    """
    dx = atom2["x"] - atom1["x"]
    dy = atom2["y"] - atom1["y"]
    dz = atom2["z"] - atom1["z"]
    return math.sqrt(dx**2 + dy**2 + dz**2)

# atoms[0] = O, atoms[1] = H1, atoms[2] = H2
r_OH1 = bond_length(atoms[0], atoms[1])
r_OH2 = bond_length(atoms[0], atoms[2])
r_HH  = bond_length(atoms[1], atoms[2])

print(f"O–H1 distance: {r_OH1:.4f} Å")
print(f"O–H2 distance: {r_OH2:.4f} Å")
print(f"H–H  distance: {r_HH:.4f} Å")
print()
print(f"Average O–H bond length: {(r_OH1 + r_OH2) / 2:.4f} Å")
print("(Experimental O–H in water: 0.9572 Å)")


#### ✏️ Practice 4.2
Write a function `all_pairwise_distances(atoms)` that computes the distance between
**every unique pair** of atoms and returns a list of tuples:
`[(label, distance), ...]` where label is like `"O1–H2"`.

Print the results sorted from shortest to longest distance.

*Hint:* Use a nested loop and only consider pairs where the second index is greater than the first.


In [ ]:
# ── Practice 4.2 ────────────────────────────────────────────────────
def all_pairwise_distances(atoms):
    """
    Return a list of (label, distance_in_angstrom) for all unique pairs.
    label format: 'Symbol+index1 – Symbol+index2', e.g. 'O1–H2'
    """
    results = []
    n = len(atoms)
    for i in range(n):
        for j in range(i + 1, n):
            # label = ...
            # dist  = bond_length(atoms[i], atoms[j])
            # results.append((label, dist))
            pass
    return results

pairs = all_pairwise_distances(atoms)
# Sort by distance
# pairs_sorted = sorted(pairs, key=lambda x: x[1])

# Print
# for label, dist in pairs_sorted:
#     print(f"  {label}: {dist:.4f} Å")


### 4.3 Computing Bond Angles

The **bond angle** at atom B in the A–B–C arrangement is found using the
**dot product** of vectors $\vec{BA}$ and $\vec{BC}$:

$$\cos\theta = \frac{\vec{BA} \cdot \vec{BC}}{|\vec{BA}|\,|\vec{BC}|}$$

$$\theta = \arccos\left(\frac{\vec{BA} \cdot \vec{BC}}{|\vec{BA}|\,|\vec{BC}|}\right)$$


In [ ]:
# ── Example 4.3 — bond angle ─────────────────────────────────────────
import math

def bond_angle(atomA, atomB, atomC):
    """
    Compute the A–B–C bond angle in degrees.
    B is the central atom.
    """
    # Vectors from B to A and from B to C
    BA = [atomA["x"] - atomB["x"],
          atomA["y"] - atomB["y"],
          atomA["z"] - atomB["z"]]

    BC = [atomC["x"] - atomB["x"],
          atomC["y"] - atomB["y"],
          atomC["z"] - atomB["z"]]

    # Dot product
    dot = BA[0]*BC[0] + BA[1]*BC[1] + BA[2]*BC[2]

    # Magnitudes
    mag_BA = math.sqrt(BA[0]**2 + BA[1]**2 + BA[2]**2)
    mag_BC = math.sqrt(BC[0]**2 + BC[1]**2 + BC[2]**2)

    # Clamp to [-1, 1] to avoid floating-point domain errors in acos
    cos_theta = max(-1.0, min(1.0, dot / (mag_BA * mag_BC)))

    return math.degrees(math.acos(cos_theta))

# H1–O–H2 angle
# atoms[0]=O, atoms[1]=H1, atoms[2]=H2
# Central atom is O (index 0)
angle_HOH = bond_angle(atoms[1], atoms[0], atoms[2])

print(f"H–O–H angle: {angle_HOH:.2f}°")
print("(Experimental H–O–H angle in water: 104.45°)")


#### ✏️ Practice 4.3
Using the `bond_angle` function:
1. Confirm the H–O–H angle from the B3LYP/6-31G* optimisation matches the experimental value (104.45°) to within 1°.
2. Write a short reflection (in a new markdown cell or as a print statement) explaining why the calculated value might differ slightly from experiment. Consider: basis set, functional, temperature, zero-point motion.


In [ ]:
# ── Practice 4.3 ────────────────────────────────────────────────────
# 1. Calculate the H-O-H angle and compare to experiment
experimental_angle = 104.45   # degrees

# angle_HOH already computed above
# difference = abs(angle_HOH - experimental_angle)
# print(f"Computed:     {angle_HOH:.2f}°")
# print(f"Experimental: {experimental_angle:.2f}°")
# print(f"Difference:   {difference:.2f}°")
# print(f"Within 1°?    {difference < 1.0}")

# 2. Add your reflection below as a comment or print statement


---
## Part 5: Putting It All Together — A Complete Parser

Now we will combine everything into a single, reusable function that reads
a Gaussian output file and returns a structured summary dictionary.


In [ ]:
# ── Example 5 — complete Gaussian output parser ──────────────────────
import math

# Atomic number → symbol look-up
ATOMIC_SYMBOLS = {
    1: "H",  5: "B",  6: "C",  7: "N",  8: "O",  9: "F",
   14: "Si", 15: "P", 16: "S", 17: "Cl", 35: "Br",
}

THERMO_KEYS = {
    "Zero-point correction=":                       "zpe_correction",
    "Thermal correction to Enthalpy=":              "thermal_enthalpy",
    "Thermal correction to Gibbs Free Energy=":     "thermal_gibbs",
    "Sum of electronic and zero-point Energies=":   "E_zpe",
    "Sum of electronic and thermal Enthalpies=":    "H_total",
    "Sum of electronic and thermal Free Energies=": "G_total",
}

def parse_gaussian_output(filename):
    """
    Parse a Gaussian output file and return a summary dict with:
      - normal_termination (bool)
      - scf_energies       (list of float, Hartree)
      - final_scf_energy   (float, Hartree)
      - thermo             (dict of thermochemical corrections, Hartree)
      - atoms              (list of {'symbol', 'x', 'y', 'z'})
    """
    result = {
        "normal_termination": False,
        "scf_energies":       [],
        "final_scf_energy":   None,
        "thermo":             {},
        "atoms":              [],
    }

    in_std_orient = False
    header_skip   = 0

    with open(filename, "r") as fh:
        lines = fh.readlines()

    for line in lines:
        # 1. Normal termination
        if "Normal termination" in line:
            result["normal_termination"] = True

        # 2. SCF energy
        if "SCF Done:" in line:
            tokens = line.split()
            result["scf_energies"].append(float(tokens[4]))

        # 3. Thermochemistry
        stripped = line.strip()
        for search_str, key in THERMO_KEYS.items():
            if stripped.startswith(search_str):
                result["thermo"][key] = float(stripped.split()[-1])

        # 4. Standard orientation
        if "Standard orientation:" in line:
            in_std_orient = True
            header_skip   = 4
            result["atoms"] = []   # reset to keep last block
            continue

        if in_std_orient:
            if header_skip > 0:
                header_skip -= 1
                continue
            if line.strip().startswith("-----"):
                in_std_orient = False
                continue
            tokens = line.split()
            anum   = int(tokens[1])
            sym    = ATOMIC_SYMBOLS.get(anum, f"Z{anum}")
            result["atoms"].append({
                "symbol": sym,
                "x": float(tokens[3]),
                "y": float(tokens[4]),
                "z": float(tokens[5]),
            })

    if result["scf_energies"]:
        result["final_scf_energy"] = result["scf_energies"][-1]

    return result

# ─────────────────────────────────────────────────────────────────────
# Run the parser and display a summary
# ─────────────────────────────────────────────────────────────────────
data = parse_gaussian_output("water.log")
H2K  = 627.509   # Hartree to kcal/mol

print("═" * 55)
print("  Gaussian Output Summary — water.log")
print("═" * 55)
print(f"  Normal termination : {data['normal_termination']}")
print(f"  Final SCF energy   : {data['final_scf_energy']:.10f} Hartree")
print()
print("  Thermochemistry:")
for k, v in data["thermo"].items():
    print(f"    {k:<40}: {v:.6f} Hartree")
print()
print("  Geometry (Standard orientation):")
print(f"    {'#':<4} {'Sym':<5} {'X':>10} {'Y':>10} {'Z':>10}")
print("    " + "-"*44)
for i, at in enumerate(data["atoms"], 1):
    print(f"    {i:<4} {at['symbol']:<5} {at['x']:>10.6f} {at['y']:>10.6f} {at['z']:>10.6f}")

# Geometry summary
if len(data["atoms"]) >= 3:
    O, H1, H2 = data["atoms"][0], data["atoms"][1], data["atoms"][2]
    rOH  = math.sqrt(sum((H1[k]-O[k])**2 for k in "xyz"))
    ang  = bond_angle(H1, O, H2)
    print()
    print(f"  O–H bond length : {rOH:.4f} Å")
    print(f"  H–O–H angle     : {ang:.2f}°")
print("═" * 55)


#### ✏️ Final Practice — Extend the Parser

Add the following features to `parse_gaussian_output`:

1. **Vibrational frequencies** — extract the list of harmonic frequencies (cm⁻¹) from lines that start with `" Frequencies --"`. Store them under key `"frequencies"`.
2. **Imaginary frequencies** — check whether any frequency is negative (imaginary). Store the count under `"n_imaginary"`. A properly optimized minimum should have zero imaginary frequencies; a transition state has exactly one.
3. Print a summary line: `"Frequencies (cm⁻¹): [1589.1, 3703.9, 3807.1]"` and `"Imaginary frequencies: 0"`.

*Hint:* A frequency line looks like:
```
 Frequencies --   1589.0765                3703.9127                3807.0960
```
After splitting, the numerical values start at index 2.


In [ ]:
# ── Final Practice ───────────────────────────────────────────────────
# Copy parse_gaussian_output here and add frequency parsing

def parse_gaussian_output_v2(filename):
    """Extended parser with vibrational frequencies."""

    # Start from the version above and add:
    # result["frequencies"] = []
    # result["n_imaginary"] = 0

    # Hint for frequency parsing:
    # if line.strip().startswith("Frequencies --"):
    #     tokens = line.split()
    #     freqs = [float(t) for t in tokens[2:]]
    #     result["frequencies"].extend(freqs)

    pass   # replace with your implementation


# Test your parser
# data2 = parse_gaussian_output_v2("water.log")
# print(f"Frequencies (cm⁻¹): {data2['frequencies']}")
# print(f"Imaginary frequencies: {data2['n_imaginary']}")


---
## Summary and Next Steps

Congratulations — you now have a working Python toolkit for reading Gaussian output files!

Here is what you learned:

| Skill | Application |
|-------|-------------|
| Variables & types | Storing atomic numbers, energies, flags |
| String methods (`split`, `strip`, `startswith`, `in`) | Identifying and parsing key lines |
| Lists & loops | Collecting multi-line data (coordinates, energies) |
| Dictionaries | Structured storage of thermochemical results |
| File I/O (`open`, `readlines`) | Reading any plain-text output file |
| Math (distance, angle) | Computing molecular geometry |

### Suggested next steps
- **Multiple molecules**: wrap `parse_gaussian_output` in a loop over a list of `.log` files to compare energies across a reaction coordinate.
- **Matplotlib**: plot SCF energy vs. optimisation step to visualise convergence.
- **Pandas**: load results into a DataFrame for tabular analysis.
- **NumPy**: use `numpy.linalg.norm` for cleaner vector arithmetic.
- **ASE / cclib**: explore dedicated computational-chemistry Python libraries built on the same concepts you used here.
